# MMDP — ניסוי ממוקד וסופי

הניסוי קבוע ואינו דורש בחירת פרמטרים:

- מפה אחת בכל רמת קושי
- 1–6 סוכנים
- Baseline RTDP מול OD-RTDP
- שני seeds מזווגים
- תכנון עד 60 שניות או עד `solved`
- evaluation של עד 5 episodes, עם תקציב מרבי של 8 שניות
- ללא diagnostics וללא conflict-risk
- transition cache מוגבל באופן זהה בשני האלגוריתמים

כל התוצאות נשמרות בקובץ יחיד:

`/content/MMDP_OUTPUT/MMDP_results.csv`

ה־CSV מכיל את ארבעת המדדים המרכזיים: זמן, זיכרון, מספר מצבים שנבדקו ושיעור הצלחה, יחד עם שדות זיהוי בסיסיים. אין הורדה אוטומטית.


In [ ]:
# משיכת קוד מ-GitHub והכנת סביבת העבודה
import subprocess, sys
from pathlib import Path
import shutil

# Clone the repository using the specific V16 branch
repo_dir = Path('/content/MMDP-OD-RTDP-PROJECT')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print('Cloning repository from GitHub...')
subprocess.run(['git', 'clone', '-b', 'v16-snapshot', 'https://github.com/t-rays/MMDP-OD-RTDP-PROJECT.git', str(repo_dir)], check=True)

PROJECT_ROOT = repo_dir
OUTPUT_DIR = Path('/content/MMDP_OUTPUT')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / 'MMDP_results.csv'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CSV:', RESULTS_CSV)


In [ ]:
# התקנת תלויות והגדרת פונקציות ההרצה והניתוח
import subprocess, sys
import pandas as pd
from IPython.display import display, Image, Markdown

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT/'requirements.txt')], check=True)

GROUP_LABELS = {'easy':'קלה', 'medium':'בינונית', 'hard':'קשה'}
GROUP_MAPS = {
    'easy': ['empty-8-8'],
    'medium': ['warehouse-10-20-10-2-1'],
    'hard': ['room-64-64-16'],
}


def run_group(group):
    print('='*80)
    print(f"מתחיל ניסוי ברמה {GROUP_LABELS[group]}: {', '.join(GROUP_MAPS[group])}")
    print('24 conditions: 1 map × 6 agent counts × 2 seeds × 2 algorithms')
    print('='*80, flush=True)
    command = [
        sys.executable, '-u', str(PROJECT_ROOT/'scripts/run_compact_matrix.py'),
        '--group', group, '--output', str(RESULTS_CSV),
    ]
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Experiment failed with code {code}')


def comparison_table(group):
    df = pd.read_csv(RESULTS_CSV)
    part = df[(df['map_group'] == group) & (df['status'] == 'ok')].copy()
    metrics = ['planning_time_seconds','planning_peak_memory_delta_mb','states_examined','success_rate']
    for col in ['n_agents', *metrics]:
        part[col] = pd.to_numeric(part[col], errors='coerce')
    means = part.groupby(['n_agents','algorithm'], as_index=False)[metrics].mean()
    wide = means.pivot(index='n_agents', columns='algorithm', values=metrics)
    result = pd.DataFrame(index=wide.index)
    names = {
        'planning_time_seconds':'time_s',
        'planning_peak_memory_delta_mb':'memory_mb',
        'states_examined':'states',
        'success_rate':'success',
    }
    for metric, short in names.items():
        result[f'{short}_baseline'] = wide[(metric,'baseline')]
        result[f'{short}_od'] = wide[(metric,'od')]
        result[f'{short}_difference_OD_minus_Baseline'] = wide[(metric,'od')] - wide[(metric,'baseline')]
    return result.reset_index().round(4)


def analyze_group(group):
    graph_dir = OUTPUT_DIR/'graphs'/group
    command = [
        sys.executable, str(PROJECT_ROOT/'scripts/analyze_compact_results.py'),
        str(RESULTS_CSV), '--group', group, '--output-dir', str(graph_dir),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    display(Markdown('## טבלת השוואה ממוצעת לפי מספר סוכנים'))
    display(comparison_table(group))
    figures = [
        ('01_time.png', 'זמן תכנון — נמוך יותר טוב'),
        ('02_memory.png', 'תוספת זיכרון בשלב התכנון — נמוך יותר טוב'),
        ('03_states_and_success.png', 'מספר מצבים שנבדקו ושיעור הצלחה'),
    ]
    for filename, title in figures:
        display(Markdown(f'### {title}'))
        display(Image(filename=str(graph_dir/filename)))
    print('\nCSV יחיד:', RESULTS_CSV)
    print('הגרפים:', graph_dir)
    print('אין הורדה אוטומטית; ניתן להוריד ידנית מחלונית Files.')


def run_and_analyze(group):
    run_group(group)
    analyze_group(group)

print('ההכנה הסתיימה. הריצו אחד משלושת תאי הקושי למטה.')


## הרצת מפה קלה

מפה: `empty-8-8`.


In [ ]:
run_and_analyze("easy")


## הרצת מפה בינונית

מפה: `warehouse-10-20-10-2-1`.


In [ ]:
run_and_analyze("medium")


## הרצת מפה קשה

מפה: `room-64-64-16`.


In [ ]:
run_and_analyze("hard")


## פלט

בתיקייה `/content/MMDP_OUTPUT` יופיעו:

- `MMDP_results.csv` — קובץ התוצאות היחיד מכל הקבוצות שהורצו
- `graphs/easy` — שלושה גרפים לרמה הקלה
- `graphs/medium` — שלושה גרפים לרמה הבינונית
- `graphs/hard` — שלושה גרפים לרמה הקשה

הרצה חוזרת של אותו תא מדלגת על conditions שכבר קיימים ב־CSV.
